# 07 · External Subset on IBM Quantum Hardware

This notebook reproduces, offline, the one IBM Quantum hardware run on the external dataset: 40 rows (20 legitimate, 20 phishing) drawn from the 6,441-row primary set of notebook 06 and classified by the frozen quantum-kernel SVM with kernels measured on `ibm_fez`. The subset, the six compared rows, the hardware settings, the metrics, and the bootstrap were frozen in `configs/external-qpu-subset.json` before any job for the subset was submitted. Here the subset is rebuilt by the frozen rule, the recorded counts are copied out of `reports/`, the six rows are re-evaluated with the project's own evaluation code, and every published number in `reports/external-qpu-subset.json` is recomputed and compared in memory. It needs no IBM account and makes no IBM Quantum call.

**Inputs**
- `configs/external-qpu-subset.json` and its parent `configs/external-validation.json` (frozen protocols, read only)
- `reports/qpu-evidence-external/<run_id>/` (`run.json`, `counts-<job>.json` for the four jobs)
- `data/external/hannousse-2020/` (from notebook 06) and `data/raw/PhiUSIIL_Phishing_URL_Dataset.csv` (from notebook 01)
- `configs/benchmark-split.json`, `configs/feature-scaler.json`, `data/processed/benchmark-sample.csv`, `artifacts/benchmark/` (trained models, training angles) from the earlier notebooks
- `reports/external-qpu-subset.json` (published values to compare against), `reports/external-validation.json` (the full external set), and `artifacts/benchmark/qpu-runs/full-ibm_fez-20261025T094010Z/evaluation.json` (the benchmark run's kernel errors, written by notebook 04)

**Outputs**
- `artifacts/benchmark/qpu-runs/<run_id>/`: copies of the recorded evidence plus `evaluation.json`, `test-kernel-raw.npy`, and `test-kernel-calibrated.npy`
- No file under `reports/`, `configs/`, or `docs/` is modified. The report is rebuilt in memory, and the last check confirms that the published report and evidence are byte-identical.

Run after: `06` (external dataset snapshot and primary set). Notebook `04` provides the benchmark run's evaluation.

Every number is labeled by source: **classical**, **exact simulation** (statevector), **IBM hardware, uncorrected**, or **IBM hardware, calibrated**. Rows are identified only by subset position, dataset `row_index`, and hashed group; no host or URL is printed.

In [1]:
import hashlib
import json
import os
import sys
from datetime import datetime
from math import ceil
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "configs").is_dir()
)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from scripts import evaluate_external as pipeline
from scripts import run_qpu_external as subset_run
from src.data.benchmark_split import group_hash
from src.evaluation import external, external_qpu
from src.evaluation.metrics import evaluate
from src.quantum import hardware, qpu_run
from src.quantum import kernel as qkernel

TOL = 1e-9
NOTEBOOK = Path("notebooks/07-external-subset-on-ibm-hardware.ipynb")
REPORT_PATH = subset_run.REPORT_PATH  # reports/external-qpu-subset.json
EVIDENCE_DIR = subset_run.EVIDENCE_DIR  # reports/qpu-evidence-external
RUNS_DIR = qpu_run.RUNS_DIR  # artifacts/benchmark/qpu-runs (Git-ignored)
BENCHMARK_RUN = "full-ibm_fez-20261025T094010Z"
SOURCE_LABELS = {
    "classical": "classical",
    "simulation": "exact simulation",
    "qpu": "IBM hardware, uncorrected",
    "qpu_calibrated": "IBM hardware, calibrated",
}


def rel(path) -> str:
    # Paths are shown relative to the repository root only.
    return Path(path).resolve().relative_to(ROOT).as_posix()


def read(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def tree_hashes(folder):
    return {rel(p): sha256(p) for p in sorted(Path(folder).rglob("*")) if p.is_file()}


def row_key(row):
    # "majority", "logistic_regression", "rbf_svm", "simulation", "qpu", "qpu_calibrated"
    return row["kernel_source"] if row["model"] == "quantum_kernel_svm" else row["model"]


def span(interval):
    return "n/a" if interval is None else f"{interval[0]:.3f}-{interval[1]:.3f}"


published = external_qpu.load(REPORT_PATH)  # validated against the report schema
RUN_ID = published["hardware"]["run_id"]
EVIDENCE = EVIDENCE_DIR / RUN_ID
# Checked at the end: this notebook never rewrites the published report or evidence.
report_bytes = REPORT_PATH.read_bytes()
evidence_before = tree_hashes(EVIDENCE_DIR)
pd.set_option("display.width", 250)
print("Repository root found; working directory set to it.")
print("Published report:", rel(REPORT_PATH), "|", published["label"], "| run", RUN_ID)
print("Recorded evidence:", sorted(p.name for p in EVIDENCE_DIR.iterdir() if p.is_dir()))

Repository root found; working directory set to it.
Published report: reports/external-qpu-subset.json | IBM Quantum hardware | run external-ibm_fez-20261025T132451Z
Recorded evidence: ['external-ibm_fez-20261025T132451Z']


## 1. The frozen protocol and the subset

`external_qpu.load_protocol` reads `configs/external-qpu-subset.json` and checks it against its parent protocol (by SHA-256) and against the hardware tooling: shots, transpiler settings, sampler, batch size, the six compared rows, and the metrics. The subset rule uses one `numpy.random.default_rng(42)` generator. It draws 20 legitimate rows, then 20 phishing rows, each with `choice` over that class's primary-set positions without replacement, and sorts all 40 by position. The rule was fixed before any result on these rows existed, so it cannot have picked rows by how the models score on them.

In [2]:
protocol = external_qpu.load_protocol(subset_run.PROTOCOL_PATH, subset_run.PARENT_PATH)
parent = read(subset_run.PARENT_PATH)
settings = protocol["hardware"]
print("Protocol:", rel(subset_run.PROTOCOL_PATH), "|", protocol["status"])
print("Frozen at", protocol["frozen_at_utc"], "| report generated", published["generated_at_utc"])
print("Subset rule:", protocol["subset"]["rule"])
pd.Series(
    {
        "backend": settings["backend"],
        "sampler": settings["sampler"],
        "shots per circuit": settings["shots"],
        "optimization level / transpiler seed": (
            f"{settings['optimization_level']} / {settings['seed_transpiler']}"
        ),
        "circuits / batch size": f"{settings['circuits']} / {settings['batch_size']}",
        "projected execution (s)": settings["projected_execution_seconds"],
        "compared rows": ", ".join(
            f"{c['model']} ({c['kernel_source']})" for c in protocol["compared"]
        ),
        "bootstrap": str(protocol["metrics"]["confidence_intervals"]),
    },
    name="configs/external-qpu-subset.json",
).to_frame()

Protocol: configs/external-qpu-subset.json | frozen before any hardware job for this subset was submitted
Frozen at 2026-10-25T12:52:39+00:00 | report generated 2026-10-25T13:34:41.535584+00:00
Subset rule: for each class in the order legitimate then phishing, numpy.random.default_rng(42).choice over that class's primary-set positions, size 20, without replacement, using one generator for both draws; then sort the 40 chosen rows by primary-set position (file order)


,configs/external-qpu-subset.json
backend,ibm_fez
sampler,v2
shots per circuit,1024
optimization level / transpiler seed,1 / 42
circuits / batch size,1008 / 252
projected execution (s),267
compared rows,"majority (classical), logistic_regression (cla..."
bootstrap,{'method': 'percentile bootstrap over the 40 r...


In [3]:
# Replication check: the report followed this protocol, frozen before the report existed.
reference = external_qpu.protocol_reference(subset_run.PROTOCOL_PATH, protocol, ROOT)
assert published["protocol"] == reference, published["protocol"]
assert published["parent_protocol"] == {
    "path": "configs/external-validation.json",
    "sha256": external.protocol_sha256(subset_run.PARENT_PATH),
}
assert protocol["parent_protocol"]["sha256"] == published["parent_protocol"]["sha256"]
assert pd.Timestamp(parent["frozen_at_utc"]) < pd.Timestamp(protocol["frozen_at_utc"])
assert pd.Timestamp(protocol["frozen_at_utc"]) < pd.Timestamp(published["generated_at_utc"])
print("OK: report protocol hash equals configs/external-qpu-subset.json:", reference["sha256"][:16])
print("OK: parent protocol hash equals configs/external-validation.json, as frozen")
print(
    "OK: parent frozen",
    parent["frozen_at_utc"],
    "< subset frozen",
    protocol["frozen_at_utc"],
    "< report generated",
    published["generated_at_utc"],
)
print("OK: shots, transpiler, sampler, batch size, compared rows and metrics match the code")

OK: report protocol hash equals configs/external-qpu-subset.json: fa3cdc9485bb6f8e
OK: parent protocol hash equals configs/external-validation.json, as frozen
OK: parent frozen 2026-10-25T11:10:26+00:00 < subset frozen 2026-10-25T12:52:39+00:00 < report generated 2026-10-25T13:34:41.535584+00:00
OK: shots, transpiler, sampler, batch size, compared rows and metrics match the code


In [4]:
# The primary set of notebook 06, then the subset by the project's own loader.
pipeline.verified_snapshot(parent["dataset"])
data, prepared, scaler = pipeline.load_inputs(parent["dataset"])
loaded = subset_run.load_subset_inputs()  # verifies, prepares, draws, featurizes and plans
subset = loaded.subset
labels = subset.labels
size = len(labels)
print(f"Primary set: {len(prepared.rows)} rows; subset: {size} rows {subset.class_counts}")
pd.DataFrame(
    {
        "primary-set position": subset.positions,
        "row_index": subset.row_index,
        "group_hash": subset.group_hash,
        "label": np.where(labels == 1, "phishing", "legitimate"),
    },
    index=pd.RangeIndex(size, name="subset position"),
)

Primary set: 6441 rows; subset: 40 rows {'legitimate': 20, 'phishing': 20}


,primary-set position,row_index,group_hash,label
subset position,,,,
0,265,353,2685d69300742759,phishing
1,422,574,8e26e9d53057f40a,phishing
2,581,814,af40464fe84a7132,legitimate
3,603,844,dcb5e8b1bf7c022d,legitimate
4,631,880,94396ed4e48c8aee,legitimate
5,870,1255,2922e1270c47004f,legitimate
6,941,1362,4f0cfb605daffd0f,phishing
7,997,1446,083df244805d3fab,phishing
8,1168,1718,17acc0a29bffd2af,phishing


In [5]:
# Replication check: the frozen rule, re-implemented directly with numpy, gives the same rows,
# and they equal the published subset and the run record's inputs.
all_labels = prepared.rows["is_phishing"].to_numpy()
per_class = protocol["subset"]["per_class"]
assert list(per_class) == ["legitimate", "phishing"]  # label 0, then label 1
rng = np.random.default_rng(published["subset"]["seed"])
draws = [
    rng.choice(np.flatnonzero(all_labels == label), size=per_class[name], replace=False)
    for label, name in enumerate(per_class)
]
positions = np.sort(np.concatenate(draws))
assert np.array_equal(positions, subset.positions)
assert prepared.rows["row_index"].to_numpy()[positions].tolist() == published["subset"]["row_index"]
groups = [group_hash(g) for g in prepared.rows["group"].to_numpy()[positions]]
assert groups == published["subset"]["group_hash"] == list(subset.group_hash)
assert all_labels[positions].tolist() == labels.tolist()
sub = published["subset"]
assert sub["size"] == protocol["subset"]["size"] == len(positions) == size
assert sub["primary_rows"] == len(prepared.rows) == loaded.primary_rows
assert sub["class_counts"] == per_class == subset.class_counts
assert sub["seed"] == external_qpu.SUBSET_SEED
_, check_angles, check_clipped = external.featurize(prepared.rows["host"].iloc[positions], scaler)
assert np.array_equal(check_angles, loaded.angles)
assert check_clipped == loaded.clipped == sub["clipped_values"]
assert published["benchmark"]["input_hashes"] == data.input_hashes
assert published["benchmark"]["model_version"] == loaded.model_version
assert published["benchmark"]["training_examples"] == len(data.train_labels)
print(f"OK: an independent default_rng({sub['seed']}) draw gives the same {size} rows (exact)")
print(f"OK: row_index, group_hash and labels equal the published subset {sub['class_counts']}")
print(f"OK: the primary set has {sub['primary_rows']} rows, as in notebook 06")
print("OK: clipped feature counts", sub["clipped_values"]["per_feature"], "equal the report")
print("OK: benchmark input hashes and model version equal the report (frozen models)")

OK: an independent default_rng(42) draw gives the same 40 rows (exact)
OK: row_index, group_hash and labels equal the published subset {'legitimate': 20, 'phishing': 20}
OK: the primary set has 6441 rows, as in notebook 06
OK: clipped feature counts {'host_entropy': 0, 'host_label_count': 4, 'host_digit_ratio': 4, 'host_hyphen_count': 0} equal the report
OK: benchmark input hashes and model version equal the report (frozen models)


## 2. The recorded hardware run

The plan is `hardware.plan_pairs(n_train=24, n_test=40)`: 960 test pairs (each subset row against each training example), then 24 calibration pairs (off-diagonal training pairs, seed 42), then 24 diagonal controls (ideal value 1). The 1,008 circuits ran on `ibm_fez` in four SamplerV2 jobs of at most 252 circuits, with 1,024 shots each. The run record holds the plan, the inputs it was planned from, the transpilation statistics, and IBM's usage metrics per job. It contains no credentials.

In [6]:
record = read(EVIDENCE / "run.json")
t = record["transpilation"]
jobs = []
for job in record["jobs"]:
    stamps = {
        key: datetime.fromisoformat(value.replace("Z", "+00:00"))
        for key, value in job["metrics"]["timestamps"].items()
    }
    usage = job["metrics"]["usage"]
    jobs.append(
        {
            "job_id": job["job_id"],
            "circuits": job["stop"] - job["start"],
            "status": job["status"],
            "queue wait (s)": round((stamps["running"] - stamps["created"]).total_seconds(), 1),
            "running -> finished (s)": round(
                (stamps["finished"] - stamps["running"]).total_seconds(), 1
            ),
            "execution (s)": job["metrics"]["circuits_execution_time_ns"] / 1e9,
            "billed (s)": usage["qpu_charge_time_seconds"],
            "resource units": usage["value"],
        }
    )
jobs = pd.DataFrame(jobs).set_index("job_id")
print(
    f"Run {record['run_id']}: backend {record['backend']}, sampler {record['sampler']}, "
    f"{record['shots']} shots, status {record['status']}"
)
print(
    f"Transpiled depth {t['depth']['min']}-{t['depth']['max']}, two-qubit gates "
    f"{t['two_qubit_gates']['min']}-{t['two_qubit_gates']['max']} "
    f"(optimization level {t['optimization_level']}, seed {t['seed_transpiler']})"
)
display(jobs.round(3))
jobs[["execution (s)", "billed (s)", "resource units"]].sum().round(3).to_frame(
    "all jobs (IBM hardware)"
)

Run external-ibm_fez-20261025T132451Z: backend ibm_fez, sampler v2, 1024 shots, status retrieved
Transpiled depth 1-54, two-qubit gates 0-10 (optimization level 1, seed 42)


,circuits,status,queue wait (s),running -> finished (s),execution (s),billed (s),resource units
job_id,,,,,,,
dbf0697gdj9s73fmk1e0,252,DONE,1.4,221.5,66.917,70,1.152
dbf069fgdj9s73fmk1g0,252,DONE,1.2,236.0,66.881,70,1.151
dbf069vsstms73fcsssg,252,DONE,90.5,244.4,66.918,70,1.152
dbf06a7pc3tc739mh380,252,DONE,94.7,345.2,66.868,70,1.151


,all jobs (IBM hardware)
execution (s),267.584
billed (s),280.000
resource units,4.607


In [7]:
# Replication check: the recorded workload is the frozen plan, and IBM's usage adds up.
assert record["plan"] == [vars(spec) for spec in loaded.plan], "recorded plan differs"
assert record["inputs"] == loaded.inputs, "the run used other inputs than this subset"
assert record["protocol"] == published["protocol"]
assert not record["pilot"] and record["status"] == "retrieved"
assert (record["backend"], record["sampler"], record["shots"]) == (
    settings["backend"],
    settings["sampler"],
    settings["shots"],
)
assert (t["optimization_level"], t["seed_transpiler"]) == (
    settings["optimization_level"],
    settings["seed_transpiler"],
)
spans = [(job["start"], job["stop"]) for job in record["jobs"]]
assert spans == hardware.batches(settings["circuits"], settings["batch_size"])
assert len(spans) == ceil(settings["circuits"] / settings["batch_size"])
assert all(job["status"] == "DONE" for job in record["jobs"])
assert [job["job_id"] for job in record["jobs"]] == published["hardware"]["job_ids"]
plan_kinds = external_qpu.plan_counts(loaded.plan)
assert plan_kinds == published["hardware"]["plan"]
assert sum(plan_kinds.values()) == settings["circuits"] == published["hardware"]["circuits"]

manual_usage = {
    "execution_seconds": sum(j["metrics"]["circuits_execution_time_ns"] for j in record["jobs"])
    / 1e9,
    "charged_seconds": float(
        sum(j["metrics"]["usage"]["qpu_charge_time_seconds"] for j in record["jobs"])
    ),
    "resource_units": sum(j["metrics"]["usage"]["value"] for j in record["jobs"]),
}
totals = qpu_run.usage_totals(record)
qpu_usage = published["runtime"]["qpu"]
for key in external_qpu.USAGE_KEYS:
    assert abs(manual_usage[key] - totals[key]) <= TOL, key
    assert abs(qpu_usage[key] - manual_usage[key]) <= TOL, key
print(
    f"OK: {settings['circuits']}-circuit plan {plan_kinds} equals qpu_run.full_plan for the subset"
)
print(
    f"OK: {len(spans)} jobs of at most {settings['batch_size']} circuits, all DONE, IDs as published"
)
print("OK: backend, sampler, shots, transpiler settings and inputs match the frozen protocol")
print(
    f"OK: IBM usage summed from run.json equals the report (1e-9): "
    f"{qpu_usage['execution_seconds']:.1f} s execution, {qpu_usage['charged_seconds']:.0f} s "
    f"billed, {qpu_usage['resource_units']:.2f} resource units"
)
projected_billed = external_qpu.projected_billed_seconds(settings["circuits"])
print(
    f"Projection before submission: {settings['projected_execution_seconds']} s execution, "
    f"{projected_billed:.1f} s billed ({external_qpu.BILLED_SECONDS_PER_CIRCUIT} s per circuit); "
    f"measured {qpu_usage['execution_seconds']:.1f} s and {qpu_usage['charged_seconds']:.0f} s"
)

OK: 1008-circuit plan {'test': 960, 'calibration': 24, 'diagonal': 24} equals qpu_run.full_plan for the subset
OK: 4 jobs of at most 252 circuits, all DONE, IDs as published
OK: backend, sampler, shots, transpiler settings and inputs match the frozen protocol
OK: IBM usage summed from run.json equals the report (1e-9): 267.6 s execution, 280 s billed, 4.61 resource units
Projection before submission: 267 s execution, 276.2 s billed (0.274 s per circuit); measured 267.6 s and 280 s


## 3. Copy the evidence and evaluate it offline

`run_qpu_external.evaluate_subset` is the function behind `scripts/run_qpu_external.py evaluate`. It writes `evaluation.json` and the kernel `.npy` files into the run directory, so the versioned evidence is first copied into the Git-ignored `artifacts/benchmark/qpu-runs/<run_id>/`; nothing is evaluated inside `reports/`. An existing copy is accepted only if it is byte-identical to the evidence. The notebook calls `evaluate_subset` directly, not the `evaluate` command, so `reports/external-qpu-subset.json` and `reports/qpu-evidence-external/` are never written; the rebuilt report stays in memory. It times the local rows over the protocol's 25 repeats, as the published report did.

In [8]:
names = ["run.json"] + [f"counts-{job['job_id']}.json" for job in record["jobs"]]
assert sorted(p.name for p in EVIDENCE.iterdir()) == sorted(names), "unexpected evidence files"
assert [p.name for p in EVIDENCE_DIR.iterdir()] == [RUN_ID], "the protocol publishes one run"
texts = {name: (EVIDENCE / name).read_text(encoding="utf-8") for name in names}
assert external_qpu.evidence_refusal(texts) is None
print(f"OK: the evidence holds run.json and {len(names) - 1} counts files only")
print("OK: no credential, instance identifier, URL or local path in the evidence")

run_dir = RUNS_DIR / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)
for name in names:
    source, destination = EVIDENCE / name, run_dir / name
    if destination.exists():
        assert sha256(destination) == sha256(source), (
            f"{rel(destination)} differs from the evidence"
        )
        status = "already present, identical"
    else:
        destination.write_bytes(source.read_bytes())
        status = "copied"
    print(f"{rel(source)} -> {rel(destination)} ({status}, sha256 {sha256(source)[:12]}...)")

OK: the evidence holds run.json and 4 counts files only
OK: no credential, instance identifier, URL or local path in the evidence
reports/qpu-evidence-external/external-ibm_fez-20261025T132451Z/run.json -> artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/run.json (already present, identical, sha256 c96c10011245...)
reports/qpu-evidence-external/external-ibm_fez-20261025T132451Z/counts-dbf0697gdj9s73fmk1e0.json -> artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/counts-dbf0697gdj9s73fmk1e0.json (already present, identical, sha256 45c08523e5f1...)
reports/qpu-evidence-external/external-ibm_fez-20261025T132451Z/counts-dbf069fgdj9s73fmk1g0.json -> artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/counts-dbf069fgdj9s73fmk1g0.json (already present, identical, sha256 a096958765b8...)
reports/qpu-evidence-external/external-ibm_fez-20261025T132451Z/counts-dbf069vsstms73fcsssg.json -> artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/counts-d

In [9]:
run_record = qpu_run.load(run_dir)
rebuilt = subset_run.evaluate_subset(
    run_dir, run_record, loaded, external_qpu.FROZEN_REPEATS, "ibm_quantum"
)
external_qpu.validate(rebuilt)
print(
    "Wrote:",
    rel(run_dir / "evaluation.json"),
    rel(run_dir / "test-kernel-raw.npy"),
    rel(run_dir / "test-kernel-calibrated.npy"),
)
table = {}
for row in rebuilt["models"]:
    entry = {}
    for name in external.METRICS:
        value = row["metrics"][name]
        point = "n/a" if value is None else f"{value:.3f}"
        entry[name] = f"{point} ({span(row['confidence_intervals'][name])})"
    entry["TN / FP / FN / TP"] = " / ".join(str(v) for r in row["confusion_matrix"] for v in r)
    table[f"{row['name']} ({SOURCE_LABELS[row['kernel_source']]})"] = entry
print(
    f"Recomputed from the counts; n={size} rows, 95% percentile bootstrap intervals in parentheses"
)
pd.DataFrame(table).T

Wrote: artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/evaluation.json artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/test-kernel-raw.npy artifacts/benchmark/qpu-runs/external-ibm_fez-20261025T132451Z/test-kernel-calibrated.npy
Recomputed from the counts; n=40 rows, 95% percentile bootstrap intervals in parentheses


,accuracy,balanced_accuracy,precision,recall,f1,average_precision,TN / FP / FN / TP
Majority baseline (classical),0.500 (0.350-0.650),0.500 (0.500-0.500),0.000 (0.000-0.000),0.000 (0.000-0.000),0.000 (0.000-0.000),n/a (n/a),20 / 0 / 20 / 0
Logistic regression (classical),0.525 (0.375-0.675),0.525 (0.370-0.675),0.533 (0.273-0.786),0.400 (0.185-0.632),0.457 (0.231-0.649),0.603 (0.412-0.803),13 / 7 / 12 / 8
RBF SVM (classical),0.525 (0.375-0.675),0.525 (0.370-0.675),0.533 (0.273-0.786),0.400 (0.185-0.632),0.457 (0.231-0.649),0.605 (0.404-0.798),13 / 7 / 12 / 8
Quantum-kernel SVM (exact simulation),0.475 (0.325-0.625),0.475 (0.325-0.625),0.480 (0.286-0.680),0.600 (0.381-0.821),0.533 (0.341-0.692),0.499 (0.339-0.734),7 / 13 / 8 / 12
"Quantum-kernel SVM (IBM hardware, uncorrected)",0.500 (0.350-0.650),0.500 (0.350-0.649),0.500 (0.304-0.696),0.650 (0.429-0.850),0.565 (0.372-0.717),0.524 (0.353-0.743),7 / 13 / 7 / 13
"Quantum-kernel SVM (IBM hardware, calibrated)",0.525 (0.375-0.675),0.525 (0.372-0.675),0.520 (0.320-0.714),0.650 (0.429-0.850),0.578 (0.378-0.731),0.504 (0.341-0.741),8 / 12 / 7 / 13


In [10]:
# Replication check: every published value, recomputed from the counts.
assert [(r["model"], r["kernel_source"]) for r in published["models"]] == list(
    external_qpu.COMPARED
)
for mine, theirs in zip(rebuilt["models"], published["models"], strict=True):
    where = f"{theirs['name']} ({theirs['kernel_source']})"
    for key in ("name", "model", "kernel_source", "confusion_matrix", "confidence_intervals"):
        assert mine[key] == theirs[key], (where, key)
    for name in external.METRICS:
        a, b = mine["metrics"][name], theirs["metrics"][name]
        assert (a is None) == (b is None) and (a is None or abs(a - b) <= TOL), (where, name)
    print(f"OK: {where:<36} metrics (1e-9), CM {mine['confusion_matrix']} and intervals (exact)")
assert rebuilt["bootstrap"] == published["bootstrap"]
assert published["bootstrap"]["resamples_used"] == published["bootstrap"]["resamples"]
for key in ("level", "resamples", "seed"):
    assert published["bootstrap"][key] == protocol["metrics"]["confidence_intervals"][key], key
same = ("source", "label", "protocol", "parent_protocol", "benchmark", "subset", "hardware")
for key in (*same, "limitations"):
    assert rebuilt[key] == published[key], key
print(
    f"OK: bootstrap {published['bootstrap']['resamples']} resamples with seed "
    f"{published['bootstrap']['seed']}, all used, as the protocol fixes"
)
print("OK: source, protocol, benchmark, subset, hardware and limitations equal the report (exact)")

kernel, mine_kernel = published["kernel"], rebuilt["kernel"]
errors = kernel["test_kernel_error_vs_ideal"]
for source in external_qpu.HARDWARE_SOURCES:
    assert abs(mine_kernel["test_kernel_error_vs_ideal"][source] - errors[source]) <= TOL, source
for key in ("mean_abs_kernel_error_vs_ideal", "diagonal_controls_mean"):
    assert abs(mine_kernel[key] - kernel[key]) <= TOL, key
cal = kernel["calibration"]
assert mine_kernel["calibration"]["valid"] is cal["valid"] is True
for key in ("a", "c", "rmse"):
    assert abs(mine_kernel["calibration"][key] - cal[key]) <= TOL, key
assert mine_kernel["calibration"]["clipped_test_values"] == cal["clipped_test_values"]
print(
    f"OK: kernel error vs exact simulation {errors['qpu']:.3f} uncorrected, "
    f"{errors['qpu_calibrated']:.3f} calibrated (1e-9)"
)
print(
    f"OK: calibration a={cal['a']:.3f}, c={cal['c']:.4f}, RMSE {cal['rmse']:.3f} (1e-9); "
    f"{cal['clipped_test_values']} clipped (exact); diagonal mean "
    f"{kernel['diagonal_controls_mean']:.3f} (1e-9)"
)

runtime_published, runtime_mine = published["runtime"], rebuilt["runtime"]
assert runtime_mine["method"] == runtime_published["method"]
assert runtime_mine["repeats"] == runtime_published["repeats"] == external_qpu.FROZEN_REPEATS
for key in external_qpu.USAGE_KEYS:
    assert abs(runtime_mine["qpu"][key] - runtime_published["qpu"][key]) <= TOL, key
for mine, theirs in zip(rebuilt["models"], published["models"], strict=True):
    a, b = mine["runtime_seconds"], theirs["runtime_seconds"]
    assert list(a) == list(b) == list(external_qpu.RUNTIME_KEYS)
    assert [v is None for v in a.values()] == [v is None for v in b.values()]
    assert all(v is None or v >= 0 for v in [*a.values(), *b.values()])
    if mine["kernel_source"] in external_qpu.HARDWARE_SOURCES:
        assert a["test_kernel"] == b["test_kernel"] == qpu_usage["execution_seconds"]
print("OK: IBM usage equals the report (1e-9); hardware test-kernel time is IBM's execution time")
print(
    "OK: runtime fields and nulls match; local runtimes are machine-dependent, checked for sanity"
)
print("  this run:  ", runtime_mine["environment"])
print("  published: ", runtime_published["environment"])

OK: Majority baseline (classical)        metrics (1e-9), CM [[20, 0], [20, 0]] and intervals (exact)
OK: Logistic regression (classical)      metrics (1e-9), CM [[13, 7], [12, 8]] and intervals (exact)
OK: RBF SVM (classical)                  metrics (1e-9), CM [[13, 7], [12, 8]] and intervals (exact)
OK: Quantum-kernel SVM (simulation)      metrics (1e-9), CM [[7, 13], [8, 12]] and intervals (exact)
OK: Quantum-kernel SVM (qpu)             metrics (1e-9), CM [[7, 13], [7, 13]] and intervals (exact)
OK: Quantum-kernel SVM (qpu_calibrated)  metrics (1e-9), CM [[8, 12], [7, 13]] and intervals (exact)
OK: bootstrap 2000 resamples with seed 42, all used, as the protocol fixes
OK: source, protocol, benchmark, subset, hardware and limitations equal the report (exact)
OK: kernel error vs exact simulation 0.026 uncorrected, 0.018 calibrated (1e-9)
OK: calibration a=0.867, c=0.0079, RMSE 0.021 (1e-9); 218 clipped (exact); diagonal mean 0.955 (1e-9)
OK: IBM usage equals the report (1e-9); hardwa

In [11]:
# Replication check, independent of the project's counts-to-kernel and metric code: P('0000')
# read straight from the counts files, np.polyfit for the calibration, scikit-learn metrics.
ideal_train = qkernel.statevector_kernel(data.train_angles)
ideal_test = qkernel.statevector_kernel(loaded.angles, data.train_angles)
raw = np.load(run_dir / "test-kernel-raw.npy")
calibrated = np.load(run_dir / "test-kernel-calibrated.npy")
raw_counts = []
for job in sorted(record["jobs"], key=lambda job: job["start"]):
    raw_counts += read(run_dir / f"counts-{job['job_id']}.json")
assert len(raw_counts) == len(record["plan"])
assert sorted({sum(c.values()) for c in raw_counts}) == published["hardware"]["shots_returned"]
manual_raw = np.full(ideal_test.shape, np.nan)
pairs, diagonal = [], []
for spec, shot_counts in zip(record["plan"], raw_counts, strict=True):
    p_zero = shot_counts.get("0000", 0) / sum(shot_counts.values())  # K = P(all zeros)
    if spec["kind"] == "test":
        manual_raw[spec["left"], spec["right"]] = p_zero
    elif spec["kind"] == "calibration":
        pairs.append((ideal_train[spec["left"], spec["right"]], p_zero))
    else:
        diagonal.append(p_zero)
assert not np.isnan(manual_raw).any() and np.array_equal(raw, manual_raw)
pair_ideal, pair_measured = np.array(pairs).T
manual_a, manual_c = np.polyfit(pair_ideal, pair_measured, 1)
manual_rmse = np.sqrt(np.mean((pair_measured - (manual_a * pair_ideal + manual_c)) ** 2))
assert abs(manual_a - cal["a"]) <= TOL and abs(manual_c - cal["c"]) <= TOL
assert abs(manual_rmse - cal["rmse"]) <= TOL
mapped = (manual_raw - manual_c) / manual_a
assert np.allclose(np.clip(mapped, 0, 1), calibrated, rtol=0, atol=TOL)
assert int(((mapped < 0) | (mapped > 1)).sum()) == cal["clipped_test_values"]
assert abs(np.mean(diagonal) - kernel["diagonal_controls_mean"]) <= TOL
assert abs(np.abs(manual_raw - ideal_test).mean() - errors["qpu"]) <= TOL
assert abs(np.abs(calibrated - ideal_test).mean() - errors["qpu_calibrated"]) <= TOL
print(
    f"OK: the {raw.shape[0]} x {raw.shape[1]} raw kernel equals P('0000') from the counts (exact)"
)
print(f"OK: np.polyfit a={manual_a:.6f}, c={manual_c:.6f} and its RMSE equal the report (1e-9)")
print(
    f"OK: calibrated kernel equals clip((raw - c) / a, 0, 1) (1e-9); "
    f"{cal['clipped_test_values']} of {raw.size} clipped"
)
print("OK: the diagonal mean and both kernel errors recomputed by hand equal the report (1e-9)")

quantum = data.models["quantum_kernel_svm"]
inputs = {
    "majority": loaded.angles,
    "logistic_regression": loaded.angles,
    "rbf_svm": loaded.angles,
    "simulation": ideal_test,
    "qpu": raw,
    "qpu_calibrated": calibrated,
}
predictions = {}
for row in published["models"]:
    model = data.models[row["model"]]
    predicted = model.predict(inputs[row_key(row)])
    score = (
        model.decision_function(inputs[row_key(row)])
        if hasattr(model, "decision_function")
        else None
    )
    predictions[row_key(row)] = predicted
    direct = {
        "accuracy": accuracy_score(labels, predicted),
        "balanced_accuracy": balanced_accuracy_score(labels, predicted),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "average_precision": None if score is None else average_precision_score(labels, score),
    }
    for name, value in direct.items():
        other = row["metrics"][name]
        assert (value is None and other is None) or abs(value - other) <= TOL, (row_key(row), name)
    assert confusion_matrix(labels, predicted, labels=[0, 1]).tolist() == row["confusion_matrix"]
print("OK: scikit-learn metrics and confusion matrices on these inputs equal the report (1e-9)")

OK: the 40 x 24 raw kernel equals P('0000') from the counts (exact)
OK: np.polyfit a=0.867041, c=0.007913 and its RMSE equal the report (1e-9)
OK: calibrated kernel equals clip((raw - c) / a, 0, 1) (1e-9); 218 of 960 clipped
OK: the diagonal mean and both kernel errors recomputed by hand equal the report (1e-9)
OK: scikit-learn metrics and confusion matrices on these inputs equal the report (1e-9)


## 4. Which predictions the hardware changed

The quantum-kernel SVM is the same trained model in all three quantum rows; only the test kernel differs. `qpu_run.measure_kernel`, the function `evaluate_subset` uses, returns the per-row hardware predictions behind the published confusion matrices. The table lists every subset row on which a hardware prediction differs from the exact-simulation prediction, by subset position and dataset `row_index`, with the SVM decision score in parentheses (positive means phishing).

In [12]:
measured = qpu_run.measure_kernel(
    run_dir, run_record, data.train_angles, loaded.angles, labels, quantum
)
by_source = {
    "simulation": evaluate(quantum, measured["ideal_test_kernel"], labels),
    **measured["evaluations"],
}
published_rows = {row_key(row): row for row in published["models"]}
for source, evaluation in by_source.items():
    assert evaluation["confusion_matrix"] == published_rows[source]["confusion_matrix"], source
    assert evaluation["predictions"] == predictions[source].tolist(), source
pred = {source: np.array(e["predictions"]) for source, e in by_source.items()}
score = {source: np.array(e["decision_scores"]) for source, e in by_source.items()}
changed = np.zeros(size, dtype=bool)
for source in external_qpu.HARDWARE_SOURCES:
    changed |= pred[source] != pred["simulation"]
flips = pd.DataFrame(
    {
        "row_index": np.array(subset.row_index)[changed],
        "label": np.where(labels[changed] == 1, "phishing", "legitimate"),
        **{
            SOURCE_LABELS[source]: [
                f"{'phishing' if p else 'legitimate'} ({v:+.3f})"
                for p, v in zip(pred[source][changed], score[source][changed], strict=True)
            ]
            for source in by_source
        },
    },
    index=pd.Index(np.flatnonzero(changed), name="subset position"),
)
flip_summary = {}
for source in external_qpu.HARDWARE_SOURCES:
    moved = pred[source] != pred["simulation"]
    flip_summary[source] = {
        "predictions changed": int(moved.sum()),
        "now correct": int((moved & (pred[source] == labels)).sum()),
        "now wrong": int((moved & (pred[source] != labels)).sum()),
        "score correlation with exact simulation": round(
            float(np.corrcoef(score["simulation"], score[source])[0, 1]), 3
        ),
    }
counts_only = ["predictions changed", "now correct", "now wrong"]
display(
    pd.DataFrame(flip_summary).T.astype(dict.fromkeys(counts_only, int)).rename(index=SOURCE_LABELS)
)
flips

,predictions changed,now correct,now wrong,score correlation with exact simulation
"IBM hardware, uncorrected",1,1,0,0.989
"IBM hardware, calibrated",2,2,0,0.987


,row_index,label,exact simulation,"IBM hardware, uncorrected","IBM hardware, calibrated"
subset position,,,,,
0,353,phishing,legitimate (-0.131),phishing (+0.137),phishing (+0.092)
5,1255,legitimate,phishing (+0.036),phishing (+0.082),legitimate (-0.054)


In [13]:
# Replication check: the changed predictions account for the accuracy differences, and every
# row's accuracy lies inside every other row's 95% accuracy interval.
accuracy = {row_key(r): r["metrics"]["accuracy"] for r in published["models"]}
interval = {row_key(r): r["confidence_intervals"]["accuracy"] for r in published["models"]}
for source in external_qpu.HARDWARE_SOURCES:
    summary = flip_summary[source]
    net = summary["now correct"] - summary["now wrong"]
    assert summary["predictions changed"] == summary["now correct"] + summary["now wrong"]
    assert abs(accuracy[source] - accuracy["simulation"] - net / size) <= TOL, source
    print(
        f"OK: {SOURCE_LABELS[source]}: {summary['predictions changed']} of {size} predictions "
        f"differ from exact simulation ({summary['now correct']} now correct, "
        f"{summary['now wrong']} now wrong), accuracy {accuracy['simulation']:.3f} -> "
        f"{accuracy[source]:.3f}"
    )
for a in accuracy:
    for b in accuracy:
        assert interval[a][0] <= accuracy[b] <= interval[a][1], (a, b)
print("OK: every row's accuracy lies inside every other row's 95% accuracy interval")

OK: IBM hardware, uncorrected: 1 of 40 predictions differ from exact simulation (1 now correct, 0 now wrong), accuracy 0.475 -> 0.500
OK: IBM hardware, calibrated: 2 of 40 predictions differ from exact simulation (2 now correct, 0 now wrong), accuracy 0.475 -> 0.525
OK: every row's accuracy lies inside every other row's 95% accuracy interval


## 5. Context: the full external set and the benchmark run

The first table sets the subset next to the same frozen models on all 6,441 primary rows (notebook 06, exact simulation only; that report has no hardware row). The second compares this run's hardware kernel with the benchmark's own `ibm_fez` run on its 12 test examples (notebook 04).

In [14]:
full_external = external.load(pipeline.REPORT_PATH)
full_rows = {r["model"]: r for r in full_external["models"]}
full_size = full_external["preparation"]["primary_rows"]
benchmark_run = read(RUNS_DIR / BENCHMARK_RUN / "evaluation.json")
assert benchmark_run["run_id"] == BENCHMARK_RUN and not benchmark_run["pilot"]
context = {}
for row in published["models"]:
    full = full_rows[row["model"]] if row["kernel_source"] in ("classical", "simulation") else None
    context[f"{row['name']} ({SOURCE_LABELS[row['kernel_source']]})"] = {
        f"subset accuracy (n={size})": (
            f"{row['metrics']['accuracy']:.3f} ({span(row['confidence_intervals']['accuracy'])})"
        ),
        f"full external accuracy (n={full_size})": (
            "n/a"
            if full is None
            else f"{full['metrics']['accuracy']:.3f} ({span(full['confidence_intervals']['accuracy'])})"
        ),
    }
display(pd.DataFrame(context).T)
pd.DataFrame(
    {
        "benchmark run, 12 x 24 (IBM hardware)": {
            "kernel error, uncorrected": benchmark_run["test_kernel_error_vs_ideal"]["qpu"],
            "kernel error, calibrated": benchmark_run["test_kernel_error_vs_ideal"][
                "qpu_calibrated"
            ],
            "calibration slope a": benchmark_run["calibration"]["a"],
            "calibration intercept c": benchmark_run["calibration"]["c"],
            "diagonal controls mean": benchmark_run["diagonal_controls_mean"],
        },
        f"external subset run, {size} x 24 (IBM hardware)": {
            "kernel error, uncorrected": errors["qpu"],
            "kernel error, calibrated": errors["qpu_calibrated"],
            "calibration slope a": cal["a"],
            "calibration intercept c": cal["c"],
            "diagonal controls mean": kernel["diagonal_controls_mean"],
        },
    }
).round(4)

,subset accuracy (n=40),full external accuracy (n=6441)
Majority baseline (classical),0.500 (0.350-0.650),0.539 (0.526-0.551)
Logistic regression (classical),0.525 (0.375-0.675),0.663 (0.651-0.674)
RBF SVM (classical),0.525 (0.375-0.675),0.644 (0.632-0.656)
Quantum-kernel SVM (exact simulation),0.475 (0.325-0.625),0.522 (0.510-0.535)
"Quantum-kernel SVM (IBM hardware, uncorrected)",0.500 (0.350-0.650),n/a
"Quantum-kernel SVM (IBM hardware, calibrated)",0.525 (0.375-0.675),n/a


,"benchmark run, 12 x 24 (IBM hardware)","external subset run, 40 x 24 (IBM hardware)"
"kernel error, uncorrected",0.0373,0.0260
"kernel error, calibrated",0.0295,0.0185
calibration slope a,0.8199,0.8670
calibration intercept c,0.0179,0.0079
diagonal controls mean,0.9244,0.9547


## 6. Interpretation

- **On 40 rows, no model separates from chance.** Accuracy is 0.500 for the majority baseline, 0.525 for logistic regression and the RBF SVM, 0.475 for the quantum-kernel SVM with the exact simulated kernel, and 0.500 and 0.525 with the uncorrected and calibrated `ibm_fez` kernels. Every row's accuracy lies inside every other row's 95% interval, and the accuracy intervals are 30 points wide (for example 0.375-0.675 for logistic regression). With 40 rows, each row is worth 2.5 points of accuracy.
- **Hardware noise changed few predictions, and that is not an improvement.** Against the exact simulation, the uncorrected hardware kernel changed 1 prediction and the calibrated one 2, and all of them happened to move toward the correct label. A difference of one or two rows of 40 is well within the intervals, so it is not evidence that hardware improves the model. Another run, with new shot noise and device drift, could move them the other way. The decision scores stay closely correlated with the simulated ones (section 4).
- **The hardware kernel was closer to the exact kernel than in the benchmark run.** Its mean error against the exact simulation is 0.026 uncorrected and 0.018 calibrated, against 0.037 and 0.029 on the benchmark run's 12 test examples (recorded in [docs/benchmark-protocol.md](../docs/benchmark-protocol.md#ibm-quantum-hardware-full-run) and rebuilt from `reports/qpu-evidence/` by notebook 04). The fresh calibration on this run's 24 training pairs has slope 0.867, intercept 0.0079 and RMSE 0.021. It clipped 218 of the 960 test values, mostly exact values near zero that noise placed below the intercept. The diagonal controls average 0.955 (ideal 1).
- **The classical models score lower on this subset than on the full external set.** Logistic regression and the RBF SVM reach 0.663 and 0.644 accuracy on all 6,441 primary rows, but 0.525 here. Their full-set accuracies still lie inside their subset intervals. The subset rule was frozen before any result on these rows existed, so the gap is sampling variation in 40 rows, not selection.
- **Cost.** The four jobs used 267.6 s of circuit execution and were billed 280 s (4.61 resource units), against a projection of 267 s and 276.2 s. That is almost half of the Open Plan's 600 s per rolling 28 days; the protocol allows this one run only.
- **Scope.** These are the frozen models, trained on 24 PhiUSIIL examples with four host-only features, evaluated once on one backend. Four qubits are trivial to simulate. **No quantum advantage is claimed.**

In [15]:
# Replication check: every number quoted in the Interpretation is formatted from the published
# files and found, as formatted, in the Interpretation cell of this notebook on disk.
interpretation = next(
    "".join(cell["source"])
    for cell in read(NOTEBOOK)["cells"]
    if "".join(cell["source"]).startswith("## 6. Interpretation")
)
lr = published_rows["logistic_regression"]
widths = {round(100 * (i[1] - i[0])) for i in interval.values() if i[1] > i[0]}
assert len(widths) == 1, widths
projected = external_qpu.projected_billed_seconds(settings["circuits"])
quoted = [
    f"{accuracy['majority']:.3f} for the majority baseline",
    f"{accuracy['logistic_regression']:.3f} for logistic regression and the RBF SVM",
    f"{accuracy['simulation']:.3f} for the quantum-kernel SVM with the exact simulated kernel",
    f"{accuracy['qpu']:.3f} and {accuracy['qpu_calibrated']:.3f} with the uncorrected and calibrated",
    f"accuracy intervals are {widths.pop()} points wide",
    f"{span(lr['confidence_intervals']['accuracy'])} for logistic regression",
    f"With {size} rows, each row is worth {100 / size:.1f} points",
    f"changed {flip_summary['qpu']['predictions changed']} prediction and the calibrated one "
    f"{flip_summary['qpu_calibrated']['predictions changed']}",
    f"{errors['qpu']:.3f} uncorrected and {errors['qpu_calibrated']:.3f} calibrated, against "
    f"{benchmark_run['test_kernel_error_vs_ideal']['qpu']:.3f} and "
    f"{benchmark_run['test_kernel_error_vs_ideal']['qpu_calibrated']:.3f}",
    f"slope {cal['a']:.3f}, intercept {cal['c']:.4f} and RMSE {cal['rmse']:.3f}",
    f"clipped {cal['clipped_test_values']} of the {raw.size} test values",
    f"average {kernel['diagonal_controls_mean']:.3f} (ideal 1)",
    f"{full_rows['logistic_regression']['metrics']['accuracy']:.3f} and "
    f"{full_rows['rbf_svm']['metrics']['accuracy']:.3f} accuracy on all {full_size:,} primary rows, "
    f"but {accuracy['rbf_svm']:.3f} here",
    f"{qpu_usage['execution_seconds']:.1f} s of circuit execution and were billed "
    f"{qpu_usage['charged_seconds']:.0f} s ({qpu_usage['resource_units']:.2f} resource units)",
    f"projection of {settings['projected_execution_seconds']} s and {projected:.1f} s",
]
missing = [text for text in quoted if text not in interpretation]
assert not missing, missing
assert accuracy["logistic_regression"] == accuracy["rbf_svm"]
for source in external_qpu.HARDWARE_SOURCES:
    assert flip_summary[source]["now wrong"] == 0, f"a {source} change moved away from the label"
for model in ("logistic_regression", "rbf_svm"):
    low, high = interval[model]
    assert accuracy[model] < full_rows[model]["metrics"]["accuracy"], model
    assert low <= full_rows[model]["metrics"]["accuracy"] <= high, model
print(f"OK: the {len(quoted)} quoted passages are formatted from the published files")
print("OK: every hardware change moved toward the correct label")
print("OK: full-set classical accuracies are higher but lie inside the subset intervals")

assert REPORT_PATH.read_bytes() == report_bytes
assert tree_hashes(EVIDENCE_DIR) == evidence_before
print("OK: reports/external-qpu-subset.json and reports/qpu-evidence-external/ were not modified")

OK: the 15 quoted passages are formatted from the published files
OK: every hardware change moved toward the correct label
OK: full-set classical accuracies are higher but lie inside the subset intervals
OK: reports/external-qpu-subset.json and reports/qpu-evidence-external/ were not modified


## 7. How the run was submitted (not executed here)

Everything above used the recorded counts. The run was submitted once, with an **IBM Quantum Open Plan account**, from the repository root in the project environment, with `IBM_QUANTUM_TOKEN`, `IBM_QUANTUM_INSTANCE`, and `IBM_QUANTUM_BACKEND=ibm_fez` set in `.env` (ignored by Git, never committed):

```powershell
python scripts/run_qpu_external.py plan                 # local: subset, 1,008 circuits, projected time
python scripts/run_qpu_external.py submit               # budget guard, transpile, confirm; 4 jobs
python scripts/run_qpu_external.py retrieve <run-id>    # repeatable; never resubmits
python scripts/run_qpu_external.py evaluate <run-id>    # local: writes reports/external-qpu-subset.json
                                                        # and copies the evidence into reports/
```

`submit` refuses to connect if any earlier run of this subset exists, because the protocol allows one run. Before submitting, it checks the instance's remaining quota against the budget guard: the projected billed time (1,008 circuits at the benchmark run's 0.274 billed seconds per circuit) plus 60 s. The instance reported 503 s remaining at submission, so about 223 s of the 600 s per rolling 28 days remained afterwards. That quota figure is printed by `submit` and is not part of the recorded evidence, so it is not checked here. A new run would produce new counts, with new shot noise and possibly a different device calibration, so its numbers would differ. The replication checks in this notebook apply only to the versioned evidence.